In [1]:
import pandas as pd

In [3]:
aqi_df = pd.read_csv("../data/processed/clean_aqi_pm25.csv")
weather_df = pd.read_csv("../data/raw/openmeteo_delhi_sample.csv")

In [9]:

weather_df.shape

(24, 6)

In [11]:
weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   datetime_local        24 non-null     str    
 1   temperature_2m        24 non-null     float64
 2   relative_humidity_2m  24 non-null     int64  
 3   wind_speed_10m        24 non-null     float64
 4   wind_direction_10m    24 non-null     int64  
 5   precipitation         24 non-null     float64
dtypes: float64(3), int64(2), str(1)
memory usage: 1.3 KB


In [12]:
weather_df.columns

Index(['datetime_local', 'temperature_2m', 'relative_humidity_2m',
       'wind_speed_10m', 'wind_direction_10m', 'precipitation'],
      dtype='str')

In [15]:
weather_df["datetime_local"].head()

0    2026-05-26T00:00
1    2026-05-26T01:00
2    2026-05-26T02:00
3    2026-05-26T03:00
4    2026-05-26T04:00
Name: datetime_local, dtype: str

In [17]:
aqi_df["datetime_local"] = pd.to_datetime(aqi_df["datetime_local"])
weather_df["datetime_local"] = pd.to_datetime(weather_df["datetime_local"])

In [20]:
aqi_df["datetime_local"].head()


0   2016-02-05 19:30:00+05:30
1   2016-02-06 10:30:00+05:30
2   2016-02-06 10:50:00+05:30
3   2016-02-06 11:30:00+05:30
4   2016-02-06 11:50:00+05:30
Name: datetime_local, dtype: datetime64[us, UTC+05:30]

In [21]:
aqi_df["datetime_local"] = aqi_df["datetime_local"].dt.tz_localize(None)

In [23]:
aqi_df.info()
weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1100 entries, 0 to 1099
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   location_id     1100 non-null   int64         
 1   location_name   1100 non-null   str           
 2   sensor_id       1100 non-null   int64         
 3   parameter       1100 non-null   str           
 4   value           1100 non-null   float64       
 5   unit            1100 non-null   str           
 6   datetime_utc    1100 non-null   str           
 7   datetime_local  1100 non-null   datetime64[us]
 8   latitude        1100 non-null   float64       
 9   longitude       1100 non-null   float64       
dtypes: datetime64[us](1), float64(3), int64(2), str(4)
memory usage: 86.1 KB
<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----     

In [24]:
aqi_df["hour"] = aqi_df["datetime_local"].dt.floor("h")
weather_df["hour"] = weather_df["datetime_local"].dt.floor("h")

In [28]:
aqi_df[["datetime_local", "hour"]].head()
    # because aqi wasnt hourly unlike weather, thus change it and check if it is now hourly
weather_df[["datetime_local", "hour"]].head()

,datetime_local,hour
0,2026-05-26 00:00:00,2026-05-26 00:00:00
1,2026-05-26 01:00:00,2026-05-26 01:00:00
2,2026-05-26 02:00:00,2026-05-26 02:00:00
3,2026-05-26 03:00:00,2026-05-26 03:00:00
4,2026-05-26 04:00:00,2026-05-26 04:00:00


In [29]:
#check if weather columns have any missing values
weather_df.isna().sum()

datetime_local          0
temperature_2m          0
relative_humidity_2m    0
wind_speed_10m          0
wind_direction_10m      0
precipitation           0
hour                    0
dtype: int64

In [30]:
weather_clean_df = weather_df[
    [
        "datetime_local",
        "hour",
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
        "wind_direction_10m",
        "precipitation"
    ]
].copy()
#hour is where the merging will happen, thus datetime_local is not needed

In [31]:
weather_clean_df.to_csv("../data/processed/clean_weather.csv", index=False)

In [32]:
#merging weatherto the aqi readings
merged_df = pd.merge(
    aqi_df,
    weather_clean_df,
    on="hour",
    how="left"
)

merged_df.head()

,location_id,location_name,sensor_id,parameter,value,unit,datetime_utc,datetime_local_x,latitude,longitude,hour,datetime_local_y,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,precipitation
0,235,"Anand Vihar, New Delhi - DPCC",384,pm25,132.0,µg/m³,2016-02-05 14:00:00+00:00,2016-02-05 19:30:00,28.646835,77.316032,2016-02-05 19:00:00,NaT,NaN,NaN,NaN,NaN,NaN
1,235,"Anand Vihar, New Delhi - DPCC",384,pm25,322.0,µg/m³,2016-02-06 05:00:00+00:00,2016-02-06 10:30:00,28.646835,77.316032,2016-02-06 10:00:00,NaT,NaN,NaN,NaN,NaN,NaN
2,235,"Anand Vihar, New Delhi - DPCC",384,pm25,188.0,µg/m³,2016-02-06 05:20:00+00:00,2016-02-06 10:50:00,28.646835,77.316032,2016-02-06 10:00:00,NaT,NaN,NaN,NaN,NaN,NaN
3,235,"Anand Vihar, New Delhi - DPCC",384,pm25,188.0,µg/m³,2016-02-06 06:00:00+00:00,2016-02-06 11:30:00,28.646835,77.316032,2016-02-06 11:00:00,NaT,NaN,NaN,NaN,NaN,NaN
4,235,"Anand Vihar, New Delhi - DPCC",384,pm25,170.0,µg/m³,2016-02-06 06:20:00+00:00,2016-02-06 11:50:00,28.646835,77.316032,2016-02-06 11:00:00,NaT,NaN,NaN,NaN,NaN,NaN


In [33]:
merged_df.shape

(1100, 17)

In [34]:
merged_df.isna().sum()

location_id                0
location_name              0
sensor_id                  0
parameter                  0
value                      0
unit                       0
datetime_utc               0
datetime_local_x           0
latitude                   0
longitude                  0
hour                       0
datetime_local_y        1100
temperature_2m          1100
relative_humidity_2m    1100
wind_speed_10m          1100
wind_direction_10m      1100
precipitation           1100
dtype: int64

In [35]:
aqi_df["hour"].min(), aqi_df["hour"].max()

(Timestamp('2016-02-05 19:00:00'), Timestamp('2025-02-20 16:00:00'))

In [36]:
weather_clean_df["hour"].min(), weather_clean_df["hour"].max()

(Timestamp('2026-05-26 00:00:00'), Timestamp('2026-05-26 23:00:00'))

In [37]:
latest_aqi_date = aqi_df["hour"].max().date()
latest_aqi_date

datetime.date(2025, 2, 20)

## Fix: Rebuild 60-day AQI + historical weather window

In [38]:
latest_aqi_time = aqi_df["hour"].max()
latest_aqi_date = latest_aqi_time.date()

start_date = (latest_aqi_time - pd.Timedelta(days=59)).date()
end_date = latest_aqi_date

start_date, end_date

(datetime.date(2024, 12, 23), datetime.date(2025, 2, 20))